# ⚙️ Feature Engineering

Feature engineering involves creating meaningful analytical variables from existing customer attributes.

The objective is to transform the cleaned telecom customer data into business-oriented features that can be used for customer segmentation, churn analysis, SQL reporting, and Power BI dashboards.

In [1]:
import pandas as pd
import numpy as np

# Load cleaned dataset
df = pd.read_csv("../data/cleaned/telco_customer_churn_cleaned.csv")

df.head()

,customer_id,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,...,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Convert Churn into a Binary Flag

In [2]:
df["churn_flag"] = df["churn"].map({
    "Yes": 1,
    "No": 0
})

df[["churn", "churn_flag"]].head()

,churn,churn_flag
0,No,0
1,No,0
2,Yes,1
3,No,0
4,Yes,1


## Customer Tenure Segment

In [3]:
df["tenure_segment"] = pd.cut(
    df["tenure"],
    bins=[-1, 6, 12, 24, 48, 60, 72],
    labels=[
        "0-6 Months",
        "7-12 Months",
        "13-24 Months",
        "25-48 Months",
        "49-60 Months",
        "61-72 Months"
    ]
)

In [4]:
df["tenure_segment"].value_counts().sort_index()

tenure_segment
0-6 Months      1470
7-12 Months      705
13-24 Months    1024
25-48 Months    1594
49-60 Months     832
61-72 Months    1407
Name: count, dtype: int64

## Monthly Charge Segment

In [5]:
df["monthly_charge_segment"] = pd.cut(
    df["monthly_charges"],
    bins=[0, 30, 60, 90, 120, np.inf],
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High",
        "Premium"
    ]
)

In [6]:
df["monthly_charge_segment"].value_counts()

monthly_charge_segment
High         2383
Very High    1739
Low          1647
Medium       1263
Premium         0
Name: count, dtype: int64

## Estimated Customer Lifetime Value

In [7]:
df["estimated_customer_value"] = (
    df["monthly_charges"] * df["tenure"]
).round(2)

In [8]:
df[
    [
        "customer_id",
        "tenure",
        "monthly_charges",
        "estimated_customer_value"
    ]
].head()

,customer_id,tenure,monthly_charges,estimated_customer_value
0,7590-VHVEG,1,29.85,29.85
1,5575-GNVDE,34,56.95,1936.30
2,3668-QPYBK,2,53.85,107.70
3,7795-CFOCW,45,42.30,1903.50
4,9237-HQITU,2,70.70,141.40


### Service Count

In [9]:
service_columns = [
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies"
]

In [10]:
df["service_count"] = (
    df[service_columns]
    .apply(lambda row: (row == "Yes").sum(), axis=1)
)

In [11]:
df["service_count"].value_counts().sort_index()

service_count
0    2213
1     966
2    1033
3    1117
4     850
5     569
6     284
Name: count, dtype: int64

### Customer Value Segment

In [12]:
df["customer_value_segment"] = pd.qcut(
    df["estimated_customer_value"],
    q=4,
    labels=[
        "Low Value",
        "Medium Value",
        "High Value",
        "Very High Value"
    ],
    duplicates="drop"
)

In [13]:
df["customer_value_segment"].value_counts()

customer_value_segment
Low Value          1759
High Value         1758
Very High Value    1758
Medium Value       1757
Name: count, dtype: int64

## New Customer Flag

In [14]:
df["new_customer_flag"] = np.where(
    df["tenure"] <= 6,
    1,
    0
)

In [15]:
df["new_customer_flag"].value_counts()

new_customer_flag
0    5562
1    1470
Name: count, dtype: int64

### High Monthly Charge Flag

In [16]:
df["high_monthly_charge_flag"] = np.where(
    df["monthly_charges"] >= df["monthly_charges"].quantile(0.75),
    1,
    0
)

### Churn Risk Segment

In [17]:
df["risk_score"] = (
    (df["contract"] == "Month-to-month").astype(int)
    + df["new_customer_flag"]
    + df["high_monthly_charge_flag"]
)

In [18]:
df["risk_segment"] = pd.cut(
    df["risk_score"],
    bins=[-1, 0, 1, 3],
    labels=[
        "Low Risk",
        "Medium Risk",
        "High Risk"
    ]
)

In [19]:
df["risk_segment"].value_counts()

risk_segment
Medium Risk    2654
Low Risk       2213
High Risk      2165
Name: count, dtype: int64

## Validate New Features

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7032 entries, 0 to 7031
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   customer_id               7032 non-null   object  
 1   gender                    7032 non-null   object  
 2   senior_citizen            7032 non-null   int64   
 3   partner                   7032 non-null   object  
 4   dependents                7032 non-null   object  
 5   tenure                    7032 non-null   int64   
 6   phone_service             7032 non-null   object  
 7   multiple_lines            7032 non-null   object  
 8   internet_service          7032 non-null   object  
 9   online_security           7032 non-null   object  
 10  online_backup             7032 non-null   object  
 11  device_protection         7032 non-null   object  
 12  tech_support              7032 non-null   object  
 13  streaming_tv              7032 non-null   object

In [21]:
df[
    [
        "customer_id",
        "churn_flag",
        "tenure_segment",
        "monthly_charge_segment",
        "estimated_customer_value",
        "service_count",
        "customer_value_segment",
        "new_customer_flag",
        "high_monthly_charge_flag",
        "risk_score",
        "risk_segment"
    ]
].head(10)


,customer_id,churn_flag,tenure_segment,monthly_charge_segment,estimated_customer_value,service_count,customer_value_segment,new_customer_flag,high_monthly_charge_flag,risk_score,risk_segment
0,7590-VHVEG,0,0-6 Months,Low,29.85,1,Low Value,1,0,2,High Risk
1,5575-GNVDE,0,25-48 Months,Medium,1936.30,2,High Value,0,0,0,Low Risk
2,3668-QPYBK,1,0-6 Months,Medium,107.70,2,Low Value,1,0,2,High Risk
3,7795-CFOCW,0,25-48 Months,Medium,1903.50,3,High Value,0,0,0,Low Risk
4,9237-HQITU,1,0-6 Months,High,141.40,0,Low Value,1,0,2,High Risk
5,9305-CDSKC,1,7-12 Months,Very High,797.20,3,Medium Value,0,1,2,High Risk
6,1452-KIOVK,0,13-24 Months,High,1960.20,2,High Value,0,0,1,Medium Risk
7,6713-OKOMC,0,7-12 Months,Low,297.50,1,Low Value,0,0,1,Medium Risk
8,7892-POOKP,1,25-48 Months,Very High,2934.40,4,High Value,0,1,2,High Risk
9,6388-TABGU,0,61-72 Months,Medium,3481.30,2,High Value,0,0,0,Low Risk


## Check for Missing Values

In [22]:
df.isnull().sum()

customer_id                 0
gender                      0
senior_citizen              0
partner                     0
dependents                  0
tenure                      0
phone_service               0
multiple_lines              0
internet_service            0
online_security             0
online_backup               0
device_protection           0
tech_support                0
streaming_tv                0
streaming_movies            0
contract                    0
paperless_billing           0
payment_method              0
monthly_charges             0
total_charges               0
churn                       0
churn_flag                  0
tenure_segment              0
monthly_charge_segment      0
estimated_customer_value    0
service_count               0
customer_value_segment      0
new_customer_flag           0
high_monthly_charge_flag    0
risk_score                  0
risk_segment                0
dtype: int64

### Save Feature-Engineered Dataset

In [24]:
df.to_csv(
    "../data/processed/telco_customer_churn_features.csv",
    index=False
)

print("✅ Feature-engineered dataset saved successfully.")

✅ Feature-engineered dataset saved successfully.
